In [0]:
df_suppliers = spark.read.table("project1.bronze.bronze_suppliers")

In [0]:
import re
from pyspark.sql.functions import col

def to_snake_case(name):
    # Substitui caracteres especiais por underscore
    name = re.sub(r'[^a-zA-Z0-9]', '_', name)
    # Adiciona underscore entre minúscula seguida de maiúscula (camelCase)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)
    # Adiciona underscore entre maiúscula seguida de maiúscula e minúscula (siglas)
    name = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', name)
    # Converte para minúsculas e remove underscores duplicados
    return re.sub(r'_+', '_', name.lower()).strip('_')

# Aplica a transformação em todas as colunas
df_suppliers_snake = df_suppliers.select(
    [col(c).alias(to_snake_case(c)) for c in df_suppliers.columns]
)

In [0]:
#Verifica quantidade de nulos na coluna
from pyspark.sql.functions import col, sum

df_suppliers_snake.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_suppliers_snake.columns
]).show()

In [0]:
from pyspark.sql.functions import col, lit, coalesce

df_suppliers_snake = (
    df_suppliers_snake
.withColumn("region",coalesce(col("region"),lit("N/A")))
.withColumn("fax",coalesce(col("fax"),lit("N/A")))
.withColumn("home_page",coalesce(col("home_page"),lit("N/A")))
)

In [0]:
df_suppliers_snake.write.format("delta").mode("overwrite").saveAsTable("project1.silver.silver_suppliers")